# Role Attribution vs Frequency

Does the model attribute more to Role when that role was common in training?

In [ ]:
import sys
from pathlib import Path
_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir(): break
    _current = _current.parent
sys.path.insert(0, str(_current))
sys.path.insert(0, str(_current / 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from collections import Counter
from tqdm.auto import tqdm

from src.interpretability.config.domestic_declarations_config import CONFIG
from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM
from src.evaluation.evaluation import Evaluation
from src.interpretability import InterpretabilityTool

%matplotlib inline

# Load
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(CONFIG.get_model_path()), dropout=0.0)
model.eval()
test_dataset = torch.load(str(CONFIG.get_test_data_path()), weights_only=False)
eval_helper = Evaluation(model=model, dataset=test_dataset, concept_name=CONFIG.concept_name,
                         growing_num_values=CONFIG.growing_num_values, all_cat=CONFIG.all_cat, all_num=CONFIG.all_num)
tool = InterpretabilityTool(model, model.data_set_categories, device='cpu')

# Find Role position
role_idx_pos = next(i for i, (n, _, _) in enumerate(model.data_set_categories[0]) if n == 'Role')
role_name = model.data_set_categories[0][role_idx_pos][0]
print(f"Cases: {len(eval_helper.cases)}, Role feature: '{role_name}' at pos {role_idx_pos}")

In [ ]:
# Count role frequencies per case
def get_role(case_name):
    case = eval_helper.cases[case_name]
    for _, prefix, _ in eval_helper._iterate_case(case):
        t = prefix[0][role_idx_pos].squeeze()
        for v in t.tolist():
            if v != 0: return v
        break
    return 0

case_roles = {c: get_role(c) for c in eval_helper.cases}
role_counts = Counter(case_roles.values())
print(f"Unique roles: {len(role_counts)}, Most common: {role_counts.most_common(5)}")

In [ ]:
# Sample cases (larger sample)
N_SAMPLES = 60
PREFIX_LENGTH = 3

np.random.seed(42)
all_cases = list(case_roles.keys())
sampled = np.random.choice(all_cases, min(N_SAMPLES, len(all_cases)), replace=False)
print(f"Sampled {len(sampled)} cases")

In [ ]:
# Compute attributions
results = []

for case_name in tqdm(sampled, desc="Computing"):
    case = eval_helper.cases[case_name]
    prefix = None
    for pl, p, _ in eval_helper._iterate_case(case):
        if pl >= PREFIX_LENGTH:
            prefix = p
            break
    if prefix is None: continue
    
    try:
        process = ([t.squeeze(0) for t in prefix[0]], [t.squeeze(0) for t in prefix[1]])
        attr_map = tool.compute_attribution_map(
            process=process, prefix_length=PREFIX_LENGTH, target=CONFIG.concept_name,
            target_class='auto', suffix_scope='step', suffix_step=0,
            method='integrated_gradients', n_steps=CONFIG.ig_steps
        )
        
        role_attr = attr_map.attributions[role_name]
        if hasattr(role_attr, 'detach'): role_attr = role_attr.detach().cpu().numpy()
        role_mag = np.abs(role_attr).sum()
        
        total_mag = sum(np.abs(v.detach().cpu().numpy() if hasattr(v, 'detach') else v).sum() 
                       for v in attr_map.attributions.values())
        
        role_idx = case_roles[case_name]
        results.append({
            'role_count': role_counts[role_idx],
            'role_attribution': role_mag,
            'role_ratio': role_mag / total_mag if total_mag > 0 else 0
        })
    except: pass

print(f"Computed: {len(results)}")

In [ ]:
# Analysis
freqs = np.array([r['role_count'] for r in results])
attrs = np.array([r['role_attribution'] for r in results])
ratios = np.array([r['role_ratio'] for r in results])

corr, p_val = stats.pearsonr(freqs, attrs)
corr_log, p_log = stats.pearsonr(np.log1p(freqs), attrs)
corr_ratio, p_ratio = stats.pearsonr(freqs, ratios)

print(f"Correlation (freq vs attribution):     r={corr:.4f}, p={p_val:.4f} {'*' if p_val<0.05 else ''}")
print(f"Correlation (log freq vs attribution): r={corr_log:.4f}, p={p_log:.4f} {'*' if p_log<0.05 else ''}")
print(f"Correlation (freq vs ratio):           r={corr_ratio:.4f}, p={p_ratio:.4f} {'*' if p_ratio<0.05 else ''}")

In [ ]:
# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(freqs, attrs, alpha=0.5)
z = np.polyfit(freqs, attrs, 1); axes[0].plot(np.sort(freqs), np.poly1d(z)(np.sort(freqs)), 'r--')
axes[0].set_xlabel('Role Frequency'); axes[0].set_ylabel('Role Attribution')
axes[0].set_title(f'Frequency vs Attribution (r={corr:.3f})')

axes[1].scatter(np.log1p(freqs), attrs, alpha=0.5)
z = np.polyfit(np.log1p(freqs), attrs, 1); axes[1].plot(np.sort(np.log1p(freqs)), np.poly1d(z)(np.sort(np.log1p(freqs))), 'r--')
axes[1].set_xlabel('Log(Frequency)'); axes[1].set_ylabel('Role Attribution')
axes[1].set_title(f'Log Freq vs Attribution (r={corr_log:.3f})')

axes[2].scatter(freqs, ratios, alpha=0.5)
z = np.polyfit(freqs, ratios, 1); axes[2].plot(np.sort(freqs), np.poly1d(z)(np.sort(freqs)), 'r--')
axes[2].set_xlabel('Role Frequency'); axes[2].set_ylabel('Attribution Ratio')
axes[2].set_title(f'Frequency vs Ratio (r={corr_ratio:.3f})')

plt.tight_layout()
plt.show()

# Conclusion
print("\n" + "="*50)
if p_val < 0.05:
    direction = "MORE" if corr > 0 else "LESS"
    print(f"SIGNIFICANT: Model attributes {direction} to Role for common roles")
else:
    print("NOT SIGNIFICANT: No clear relationship between role frequency and attribution")